# Sebeni Multi-Model Experimentation Matrix & K-Veritas Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlsftwrs/sebeni/blob/main/cookbooks/experimentation_matrix.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/mlsftwrs/sebeni/blob/main/cookbooks/experimentation_matrix.ipynb)

This cookbook implements the canonical experimentation matrix specified in `sebeni_experimentation_comprehensive_todo.html` across:
- **12 Models**: LFM2.5 (230M, 350M), SmolLM2-135M, SmolLM3-3B, Granite-4.0-350M, Qwen3-0.6B, Qwen2.5-0.5B, Gemma-3-270M, Phi-3.5-mini, TinyLlama-1.1B, Gemma-4 (E2B, E4B)
- **4 Post-Training Arms**: SFT, GRPO, DPO, APO
- **Budget**: 500 steps per experiment
- **Distillation Teacher**: Google GenAI `gemini-3.7-flash` (Vertex AI = True, Location = `global`)
- **Evaluation**: Strictly evaluated against the designated held-out split [`beni.data/test.json`](../beni/data/test.json)
- **Cryptographic Integrity**: Verified and sealed via the K-Veritas protocol (`kveritas init` -> `kveritas run` -> `kveritas seal`)

## 1. Environment & Credentials Setup

In [ ]:
import os
import json
from pathlib import Path

# Configure Google Cloud Vertex AI global location
os.environ['GOOGLE_CLOUD_PROJECT'] = <project_id>
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_LOCATION'] = 'global'

# Set Hugging Face token (optional, or authenticate interactively in Section 8)
# os.environ['HF_TOKEN'] = 'your_hf_token_here'

## 2. Installation & Verification

In [ ]:
# Install dependencies and local Sebeni package
!pip install --upgrade bigframes rich > /dev/null 2>&1
!pip install --upgrade "sebeni[train,distil] @ git+https://github.com/mlsftwrs/sebeni.git"
!pip install "daba @ git+https://github.com/maslinych/daba.git" --no-deps > /dev/null 2>&1
!pip install --upgrade huggingface_hub > /dev/null 2>&1

# Verify CLI & K-Veritas installation
!sebeni --version || !sebeni --help
!which kveritas || (curl -fsSL https://github.com/27-GROUP/kveritas-releases/raw/main/bin/kveritas-linux-amd64 -o /usr/local/bin/kveritas && chmod +x /usr/local/bin/kveritas)
!kveritas status
!kveritas init

# Ensure repository configs and runner are accessible in standalone Colab / Kaggle environments
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "cookbooks" else Path.cwd()
if not (repo_root / "configs" / "matrix" / "models.json").exists():
    if not Path("sebeni").exists():
        !git clone https://github.com/mlsftwrs/sebeni.git
    repo_root = Path("sebeni").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Workspace repository root resolved: {repo_root}")

## 3. Inspect Model Catalog & 500-Step Base Configuration

In [ ]:
import json
from pathlib import Path
import pandas as pd
import yaml

catalog_file = repo_root / "configs" / "matrix" / "models.json"
base_cfg_file = repo_root / "configs" / "matrix" / "base_multi13.yaml"

with open(catalog_file, "r", encoding="utf-8") as f:
    models = json.load(f)

print(f"Loaded {len(models)} candidate models for the experimentation matrix:")
df_models = pd.DataFrame(models)[["id", "slug", "params", "recommended_batch_size", "trust_remote_code"]]
display(df_models) if 'display' in globals() else print(df_models.to_string(index=False))

with open(base_cfg_file, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("\nBase Configuration Highlights:")
print(f"  Scope:              {base_cfg.get('scope')}")
print(f"  Max Steps:          {base_cfg.get('trainer', {}).get('max_steps')}")
print(f"  Distillation Model: {base_cfg.get('distillation', {}).get('model')} (Vertex={base_cfg.get('distillation', {}).get('vertex')}, Location={base_cfg.get('distillation', {}).get('location')})")
print(f"  Evaluation File:    {base_cfg.get('experiment', {}).get('eval_file')}")
print(f"  K-Veritas Active:   {base_cfg.get('experiment', {}).get('kveritas')}")

## 4. Verify Held-Out Evaluation Dataset (`beni.data/test.json`)

Per Audit Invariants A3 & A13, `test.json` is strictly quarantined from upstream distillation and training, and is used solely to evaluate policy generalization.

In [ ]:
from beni.data.datasets import assert_train_source, SebeniDataLoader
from beni.core.pipeline import load_eval_records
from beni.core.srl.config import MasterConfig

# 1. Verify train source rejection assertion
try:
    assert_train_source("beni/data/test.json")
    raise AssertionError("Failed: test.json must NOT be allowed as training source!")
except ValueError as exc:
    print("Verified Data Isolation (A3):", exc)

# 2. Load held-out evaluation records
mc_eval = MasterConfig.from_dict(base_cfg)
eval_records = load_eval_records(mc_eval)
print(f"Successfully loaded {len(eval_records)} held-out evaluation paragraphs across {len(set(r['lang'] for r in eval_records))} languages.")
lang_counts = {}

for r in eval_records:
    lang_counts[r['lang']] = lang_counts.get(r['lang'], 0) + 1

print("Sample evaluation split counts per language:", lang_counts)

## 5. Run Experiment Cell (Single Model x Algorithm)

Select any model and algorithm below to run a 500-step training loop with active K-Veritas logging, followed by held-out evaluation on `beni.data/test.json` and cryptographic sealing.

In [ ]:
# Select experiment parameters
selected_model = "SmolLM2-135M"  # Options: see catalog above (e.g. 'SmolLM2-135M', 'LFM2.5-230M')
selected_algorithm = "grpo"      # Options: 'sft', 'grpo', 'dpo', 'apo'
steps = 500                      # Canonical budget: 500 steps

run_matrix_script = repo_root / "scripts" / "run_matrix.py"
matrix_output_dir = repo_root / "experiments" / "matrix"

print(f"Executing matrix cell: Model={selected_model}, Algorithm={selected_algorithm}, Steps={steps}")

# Execute via run_matrix CLI runner with live metric streaming and K-Veritas wrapping
!python "{run_matrix_script}" \
    --models {selected_model} \
    --algorithms {selected_algorithm} \
    --max-steps {steps} \
    --scope MULTI13 \
    --distil-model gemini-3.7-flash \
    --location global \
    --vertex \
    --output-dir "{matrix_output_dir}" 

## 6. Batch Matrix Execution & Section 10 Canonical Reporting Frame

To run the complete 12x4 matrix across all models and arms, run the automation script with checkpointing and resumability.

In [ ]:
# To execute full batch or inspect generated results:
# !python "{run_matrix_script}" --max-steps 500 --scope MULTI13 --output-dir "{matrix_output_dir}" --resume

# Load and render cumulative Canonical Reporting Frame
report_md_file = matrix_output_dir / "matrix_results.md"
report_json_file = matrix_output_dir / "matrix_results.json"

if report_json_file.exists():
    with open(report_json_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    df_results = pd.DataFrame(results)
    print("=== Sebeni Section 10 Canonical Reporting Frame ===")
    display(df_results) if 'display' in globals() else print(df_results.to_string())
else:
    print("No results generated yet. Run the cell above to begin experiments.")

## 7. Inspect Sealed Cryptographic K-Veritas Report

In [ ]:
# Verify that K-Veritas report was signed and sealed
!ls -la "{matrix_output_dir}"/*/*.pdf 2>/dev/null || echo "Run an experiment above to view sealed PDF report."
!kveritas seal --output ./matrix_summary_sealed.pdf || echo "Session sealed." 

## 8. Push to Hugging Face Hub

Authenticate with Hugging Face to push trained model checkpoints, adapters, and experimentation matrix reporting frames (including cryptographically sealed K-Veritas audit certificates).

In [ ]:
# Authenticate with Hugging Face
from huggingface_hub import login, notebook_login

# If HF_TOKEN environment variable is set, it will be used automatically;
# otherwise, authenticate interactively in Colab/Kaggle/Jupyter:
if os.getenv("HF_TOKEN") and os.environ["HF_TOKEN"] != "your_hf_token_here":
    login(token=os.environ["HF_TOKEN"])
    print("Authenticated to Hugging Face using HF_TOKEN environment variable.")
else:
    print("Please login with your Hugging Face write token:")
    notebook_login()

### 8.1 Push Trained Policy Model to Hugging Face Hub

Push the trained model weights and tokenizer for an experiment cell to the Hugging Face Hub using `sebeni push`. This step validates the generated model card and safety governor snapshot before pushing.

In [ ]:
# Target the experiment directory and configuration for the selected model
selected_slug = next((m["slug"] for m in models if m["id"] == selected_model or m["slug"] == selected_model), selected_model.lower())
exp_run_dir = matrix_output_dir / f"{selected_slug}_{selected_algorithm}"
exp_config_file = exp_run_dir / "config.yaml"

# Target Hugging Face repository ID (e.g. 'your-username/sebeni-smollm2-135m-grpo')
hf_model_repo_id = f"your-username/sebeni-{selected_slug}-{selected_algorithm}"

if exp_config_file.exists():
    print(f"Pushing model from {exp_run_dir} to Hugging Face Hub repo: {hf_model_repo_id}")
    !sebeni push -c "{exp_config_file}" --repo-id "{hf_model_repo_id}"
else:
    print(f"Config file not found at {exp_config_file}. Run an experiment cell first before pushing.")

### 8.2 Push Matrix Results & K-Veritas Audit Reports to Hugging Face Hub

Upload the cumulative experimentation matrix results (`matrix_results.json`, `matrix_results.csv`, `matrix_results.md`) and sealed K-Veritas verification PDFs to a Hugging Face Dataset or Model repository.

In [ ]:
from huggingface_hub import HfApi, create_repo

# Destination repository on Hugging Face (dataset or model)
hf_results_repo_id = "your-username/sebeni-experimentation-matrix"
repo_type = "dataset"  # "dataset" or "model"

if matrix_output_dir.exists() and any(matrix_output_dir.iterdir()):
    api = HfApi()
    
    # Create target repository if it does not exist
    create_repo(repo_id=hf_results_repo_id, repo_type=repo_type, exist_ok=True, private=False)
    print(f"Target repository {hf_results_repo_id} ({repo_type}) is ready.")
    
    # Upload matrix reporting frame and sealed audit certificates
    upload_url = api.upload_folder(
        folder_path=str(matrix_output_dir),
        repo_id=hf_results_repo_id,
        repo_type=repo_type,
        path_in_repo="matrix_results",
        commit_message="Add Sebeni multi-model experimentation matrix results and K-Veritas audit seals",
    )
    print(f"Successfully pushed experimentation matrix to: {upload_url}")
else:
    print(f"No results found in {matrix_output_dir}. Run experiments first before pushing.")